In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_cleaned;
use schema silver_cleaned;
select current_catalog(), current_schema();

In [0]:
loans_defaulter_df = spark.read.table('bronze.LoanDefaulter')

In [0]:
#ingested date

loan_d_ingested_df = loans_defaulter_df.withColumn('ingested_date', current_timestamp())

In [0]:
#few delinq_2yrs values are like 20.66 so it should be 21 - we need whole number 
#also null values into 0

loan_d_updated_df = loan_d_ingested_df.withColumn('delinq_2yrs',
                                                 col('delinq_2yrs').cast('integer'))\
.fillna(0, subset=['delinq_2yrs'])

In [0]:
loan_d_updated_df.groupby('delinq_2yrs').count().sort(desc('count')).show(40)

In [0]:
#like delinq_2yrs perform same cleaning for set of data

#public_rec
loan_d_pr_df = loan_d_updated_df.withColumn('public_rec',
                                                 col('public_rec').cast('integer'))\
.fillna(0, subset=['public_rec'])

In [0]:
#public_bankruptcies

loan_d_pb_df = loan_d_pr_df.withColumn('public_bankruptcies',
                                                 col('public_bankruptcies').cast('integer'))\
.fillna(0, subset=['public_bankruptcies'])

In [0]:
#inquiry_6_months

loan_d_inq_df = loan_d_pb_df.withColumn('inquiry_6_months',
                                                 col('inquiry_6_months').cast('integer'))\
.fillna(0, subset=['inquiry_6_months'])

In [0]:
#display(loan_d_inq_df.head(2))

In [0]:
#convert months_since_last_delinq and months_since_last_public_record into integer

loan_d_months_last_delinq_df = loan_d_inq_df.withColumn('months_since_last_delinq', col('months_since_last_delinq').cast('integer'))\
.fillna(0, subset=['months_since_last_delinq'])

loan_d_months_last_pub_rec_df = loan_d_months_last_delinq_df.withColumn('months_since_last_public_record', col('months_since_last_public_record').cast('integer'))\
.fillna(0, subset=['months_since_last_public_record'])

In [0]:
#logic for defaulter list

loan_delinq_defaulter_df = loan_d_months_last_pub_rec_df.select('member_id','delinq_2yrs','delinq_amount','months_since_last_delinq')\
.filter("delinq_2yrs > 0 or months_since_last_delinq > 0")

In [0]:
loan_delinq_defaulter_df.count()

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_delinq/", recurse=True)

In [0]:
loan_delinq_defaulter_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_delinq/')

In [0]:
%sql
create or replace table silver_cleaned.loans_defaulter_delinq
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_delinq/`

In [0]:
#logic for public record list

loan_public_record_defaulter_df = loan_d_months_last_pub_rec_df.select('member_id','public_rec','public_bankruptcies','inquiry_6_months','months_since_last_public_record')

In [0]:
#list of members who are having public records

loan_public_rec_member_df = loan_public_record_defaulter_df.select('member_id')\
.filter("public_rec > 0 or public_bankruptcies > 0 or inquiry_6_months >0")

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_public_record/", recurse=True)

In [0]:
loan_public_record_defaulter_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_public_record/')

In [0]:
%sql
create or replace table silver_cleaned.loans_defaulter_public_record
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/loans_defaulter_public_record/`

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/loans_public_defaulter_member/", recurse=True)

In [0]:
loan_public_rec_member_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/loans_public_defaulter_member/')

In [0]:
%sql
create or replace table silver_cleaned.loans_public_defaulter_member
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/loans_public_defaulter_member/`